In [ ]:
#réinitialise module

%load_ext autoreload
%autoreload 2


import os

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"

# Le réglage de la mémoire JVM (équivalent Python de options(java.parameters = "-Xmx2G"))
# est fait en cellule 1, avant `import r5py` — voir les commentaires là-bas.
# Il doit rester avant cet import car, en Python, la JVM démarre dès `import r5py`
# (contrairement à R où `library(r5r)` seul ne la démarre pas).


# Fix : forcer l'initialisation de PROJ/GDAL avec les données de rasterio
# AVANT import r5py. Le démarrage de la JVM par r5py écrase la variable
# d'environnement PROJ_LIB avec un vieux chemin Miniconda inexistant
# (/opt/miniconda3/share/proj), ce qui casse ensuite toute reprojection
# rasterio/contextily ("Cannot find proj.db"). Importer rasterio ici initialise
# le contexte PROJ de GDAL avec le proj.db du venv avant que r5py ne le pollue ;
# ce contexte est mis en cache pour tout le process, donc l'ordre suffit.
import rasterio

import r5py
import r5py.util.jvm
r5py.util.jvm.MAX_JVM_MEMORY = 2 * 1024**3  # 2 Go (réduit de 3 Go : machine 16 Go RAM sous forte pression mémoire, cf. kernel mort du 21/07). À remonter si vous avez libéré de la RAM (fermé VS Code/Chrome) et traitez une grosse agglo (Toulouse, Lyon...).
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib
import branca as bc
import folium
import pyarrow
import pyarrow.parquet
#import r5py.sampledata.helsinki 

from src.info_reseau import dates_service, nom_reseau, nom_reseau_str
import gtfs_kit as gk

from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    km_par_ligne_plage,
    obtenir_service_ids_pour_date,
    exporter_df_to_csv,
    exporter_geojson,
    exporter_gdf_to_csv,
    dir_tree,
    
)

from pathlib import Path
import datetime
import re
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import json
import time
import requests

from src.build_data_agglo import (
    codes_communes_via_api,
    details_communes,
    build_decoupage_agglo,
    decoupage_agglo_geojson,
    osm_pbf_creator,
    build_grid_agglo,
)

from src.BPE_traitement import (
    import_BPE, 
    filtre_BPE, 
    filtre_BPE_actifs, 
    carte_ponderation_domaine,
    mask_domaine_bpe,
   land_use_data_domaine,  
)

from src.utilitaires_matrix import (
    cumulative_cutoff,
    cost_to_closest,
    decay_exponential,
    gravity,
    enhanced_2sfca,  
)    


#chemins fixe 

BASE_DIR = os.getcwd()  # Remonte d'un niveau depuis scripts/

BPE_XLS_PATH=os.path.join(BASE_DIR,'data', "BPE_gammes_equipements_2025.xlsx")

MEMORY_CSV_AGGLO_DIR = os.path.join(BASE_DIR, "data", "memory_csv_agglo")

output_path=os.path.join(BASE_DIR,'output')

data_path=os.path.join(BASE_DIR,'data')




In [ ]:

#import BPE 

BPE_PATH=os.path.join(BASE_DIR,'data',"BPE25.parquet") # base de données BPE https://catalogue-donnees.insee.fr/fr/catalogue/recherche/DS_BPE 2025

BPE_URL = "https://www.insee.fr/fr/statistiques/fichier/8217525/BPE25.parquet"

import_BPE(BPE_URL) 

In [ ]:
#chemin GTFS 

GTFS_PATH=os.path.join(BASE_DIR,"data","GTFS","MEL_gtfs.zip")


#charger le gtfs pour définir date 
feed = charger_gtfs(GTFS_PATH)


#ajoute un text pour les GTFS régionaux qui ne peuvent pas être traités  

nb_agences = len(feed.agency)
if nb_agences > 3:
    print(f"⚠ Ce GTFS regroupe {nb_agences} agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement. Le programme s'arrête")

 #cherche nom réseau 
    
nom_reseau_str=nom_reseau_str(feed)


In [ ]:

# créer un fichier decoupage_agglo.csv à partir d'un GTFS quelconque

decoupage_reference_path_reseau=os.path.join(BASE_DIR, "data","memory_csv_agglo",f"decoupage_agglo_{nom_reseau_str}.csv") 


# Si aucun decoupage_agglo n'a encore été sauvegardé pour ce réseau (nom_reseau_str),
# le fichier n'existe pas : build_decoupage_agglo() plante sinon en essayant de le lire.
if not os.path.exists(decoupage_reference_path_reseau):
    decoupage_reference_path_reseau = None


decoupage_agglo = build_decoupage_agglo(
    gtfs_path=GTFS_PATH,
    output_path=os.path.join(BASE_DIR, "data", "decoupage_agglo.csv"),
    decoupage_reference_path=decoupage_reference_path_reseau,  # optionnel : accélère si déjà connu
)
decoupage_agglo 


# Sauvegarde de mémoire : conserve un decoupage_agglo par réseau (nom_reseau_str),
# pour ne pas écraser celui d'un autre GTFS déjà traité.

os.makedirs(MEMORY_CSV_AGGLO_DIR, exist_ok=True)
exporter_df_to_csv(
    decoupage_agglo,
    os.path.join(MEMORY_CSV_AGGLO_DIR, f"decoupage_agglo_{nom_reseau_str}.csv"),
)

DECOUPAGE_COM_PATH_CSV = os.path.join(BASE_DIR, "data", "decoupage_agglo.csv")

# créer un fichier geojson à partir de decoupage_agglo.csv

decoupage_agglo_gdf = decoupage_agglo_geojson(csv_path=DECOUPAGE_COM_PATH_CSV)

DECOUPAGE_COM_PATH_GEOJSON = os.path.join(BASE_DIR, "data", "decoupage_agglo.geojson")

# créer un fichier osm_pbf à partir de decoupage_agglo.geojson

agglo_osm_pbf = osm_pbf_creator(DECOUPAGE_COM_PATH_GEOJSON)

DECOUPAGE_COM_PATH_PBF = os.path.join(BASE_DIR, "data", "agglo.osm.pbf")


In [ ]:
# pour éviter de faire tourner la cellule ci dessus chronophage 

DECOUPAGE_COM_PATH_PBF = os.path.join(BASE_DIR, "data", "agglo.osm.pbf")
DECOUPAGE_COM_PATH_GEOJSON = os.path.join(BASE_DIR, "data", "decoupage_agglo.geojson")
DECOUPAGE_COM_PATH_CSV = os.path.join(BASE_DIR, "data", "decoupage_agglo.csv") 

# créer un fichier grid 200x200

population_grid_agglo = build_grid_agglo(DECOUPAGE_COM_PATH_GEOJSON)

GEO_GPKG_PATH = os.path.join(BASE_DIR, "data", "population_grid_agglo.gpkg")

In [ ]:



#Retourne différentes dates 
#dates_service, date_debut , date_fin , date_JOB = dates_service(feed)

dates_service_list, date_debut, date_fin, date_JOB = dates_service(feed)


#Carroyage 200x200 INSEE https://www.insee.fr/fr/statistiques/8272002

population_grid_agglo = gpd.read_file(GEO_GPKG_PATH)

# défini land_use_data surlequel on va travailler par a suite 

land_use_data = population_grid_agglo[["id", "population"]].copy()

#défini réseau de transport, argument elevation non nécessaire n'a pas été mis 

from src.utils import preparer_gtfs_pour_r5py

GTFS_PATH_R5PY = preparer_gtfs_pour_r5py(GTFS_PATH)

transport_network = r5py.TransportNetwork(
    osm_pbf=DECOUPAGE_COM_PATH_PBF,
    gtfs=[GTFS_PATH_R5PY],
)
population_grid_agglo.info()

#affiche carte agglo avec carroyage 200x200

overview_map_agglo = population_grid_agglo.explore(
    "population",
    cmap="Reds",
    tiles="OpenStreetMap",
    style_kwds={
        "style_function": lambda x: {
            "fillOpacity": 0 if x["properties"]["population"] == 0 else 0.7,
            "weight": 0,
            "opacity": 0,
        }
    },
)
#overview_map_aggl
overview_map_agglo



In [ ]:
#analyse BPE  1.1


# BPE25.parquet (INSEE) est un parquet tabulaire classique (colonnes LONGITUDE/
# LATITUDE, pas de géométrie WKB/métadonnées GeoParquet) : gpd.read_parquet()
# échoue avec "Missing geo metadata". pd.read_parquet() est la bonne fonction ici.


# decoupage_agglo.csv est plus simple que le .geojson ici : il suffit d'une jointure
# attributaire sur le code commune INSEE (pas besoin de jointure spatiale avec
# reprojection). code_insee est un int (ex: 17300) ; DEPCOM dans le BPE est une
# chaîne de 5 caractères (ex: "17300") : on caste et on zero-pad pour faire matcher.


# Liste des types d'équipements présents (TYPEQU = code le plus fin de la
# nomenclature BPE, ex: "C1", "D201"... ; le parquet ne contient pas les libellés
# associés, seulement les codes). DOM/SDOM donnent des catégories plus larges
# (domaine / sous-domaine) si TYPEQU est trop détaillé pour ton usage.


# Rattachement de BDE_cda au carroyage population_grid_cda : jointure spatiale.
# LAMBERT_X/LAMBERT_Y du BPE sont déjà en EPSG:2154 (vérifié : identique à la CRS
# de population_grid_cda), donc pas besoin de reprojection.


BPE_agglo=filtre_BPE (DECOUPAGE_COM_PATH_CSV,population_grid_agglo)


# Pondération de land_use_data par gamme d'équipement (BPE_gammes_equipements_2025.xlsx).
# Chaque TYPEQU appartient à une gamme (proximité/intermédiaire/supérieure/hors gamme),
# reflétant sa rareté/importance ; on pondère chaque équipement de BDE_cda par le poids
# de sa gamme plutôt que de compter chaque équipement à l'identique.
#
# Note : ~2,4% des équipements de la zone (codes F1xx/F2xx/G10x, domaines
# sport/tourisme) n'ont pas de gamme dans la nomenclature officielle INSEE et
# sont donc exclus du score pondéré (mais restent comptés dans "TYPEQU" plus haut).

#proposition de pondération par gammes cf BPE_gammes_equipements_2025.xlsx dans le dossier data et file:///Users/antoinechevre/Desktop/Dossier_index/Data/BPE25_liste_hierarchisee_TYPEQU.html


gammes_poids_A = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 3,
    "Gamme supérieure": 4,
    "Hors Gamme": 3,
}

gammes_poids_B = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 4,
    "Gamme supérieure": 6,
    "Hors Gamme": 8,
}

gammes_poids_C = {
    "Gamme de proximité": 4,
    "Gamme intermédiaire": 6,
    "Gamme supérieure": 8,
    "Hors Gamme": 10,
}

gammes_poids_D = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 4,
    "Gamme supérieure": 6,
    "Hors Gamme": 8,
}

gammes_poids_E = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 4,
    "Gamme supérieure": 6,
    "Hors Gamme": 8,
}

gammes_poids_F = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 4,
    "Gamme supérieure": 6,
    "Hors Gamme": 8,
}

gammes_poids_G = {
    "Gamme de proximité": 2,
    "Gamme intermédiaire": 4,
    "Gamme supérieure": 6,
    "Hors Gamme": 8,
}

gamme_typequ = pd.read_excel(
    BPE_XLS_PATH,
    sheet_name="Gammes 2025 1 ligne 1 Typequ",
    header=4,
)[["TYPEQU", "GAMME"]]

BPE_agglo = BPE_agglo.merge(gamme_typequ, on="TYPEQU", how="left")

# Poids par gamme spécifique à chaque grand domaine BPE (A-G, première lettre de
# TYPEQU) plutôt qu'une seule grille commune : on peut ainsi valoriser une même
# gamme différemment selon le domaine (ex: "Hors Gamme" pèse 4 en A mais 8
# ailleurs, cf. gammes_poids_A ci-dessus).
gammes_poids_par_domaine = {
    "A": gammes_poids_A,
    "B": gammes_poids_B,
    "C": gammes_poids_C,
    "D": gammes_poids_D,
    "E": gammes_poids_E,
    "F": gammes_poids_F,
    "G": gammes_poids_G,
}

table_poids_domaine_gamme = pd.DataFrame(
    [
        {"domaine": domaine, "GAMME": gamme, "poids_gamme": poids}
        for domaine, poids_par_gamme in gammes_poids_par_domaine.items()
        for gamme, poids in poids_par_gamme.items()
    ]
)

BPE_agglo["domaine"] = BPE_agglo["TYPEQU"].str[0]
BPE_agglo = BPE_agglo.merge(table_poids_domaine_gamme, on=["domaine", "GAMME"], how="left")

equipements_pondere_par_carreau = (
    BPE_agglo.dropna(subset=["id_carreau", "poids_gamme"])
    .groupby("id_carreau")["poids_gamme"]
    .sum()
)

land_use_data["equipements_pondere"] = (
    land_use_data["id"].map(equipements_pondere_par_carreau).fillna(0.0)
)

# "Pôles d'équipements" : carreaux dont le score pondéré dépasse la moyenne de
# la zone d'un certain pourcentage. SEUIL_POLE_EQUIPEMENTS_PCT est le paramètre
# à ajuster pour affiner/customiser le seuil plus tard (1.30 = 30% au-dessus
# de la moyenne).

SEUIL_POLE_EQUIPEMENTS_PCT = 1.30

seuil_equipements_pondere = (
    SEUIL_POLE_EQUIPEMENTS_PCT * land_use_data["equipements_pondere"].mean()
)

land_use_data["pole_equipements"] = (
    land_use_data["equipements_pondere"] > seuil_equipements_pondere
).astype(int)

print(
    f"{land_use_data['pole_equipements'].sum()} carreaux au-dessus du seuil "
    f"({seuil_equipements_pondere:.1f}, soit {SEUIL_POLE_EQUIPEMENTS_PCT:.0%} de la moyenne)"
)


# Filtres land_use_data par domaine BPE (source :
# BPE25_liste_hierarchisee_TYPEQU.html) : un DataFrame land_use_data_<lettre>
# par domaine, avec le nombre d'équipements de ce domaine par carreau.
# TYPEQU commence toujours par la lettre du domaine (ex: "C107" -> domaine C),
# donc un simple str.startswith(lettre) suffit.
DOMAINES_BPE = {
    "O": "Tout équipements pondérés",
    "A": "Services pour les particuliers",
    "B": "Commerces",
    "C": "Enseignement",
    "D": "Santé et action sociale",
    "E": "Transports et déplacements",
    "F": "Sports, loisirs et culture",
    "G": "Tourisme",
}

In [ ]:
#analyse BPE 1.2

# Ne garder que les carreaux "actifs" pour l'analyse BPE : ceux qui ont de la
# population ou au moins un équipement pondéré (equipements_pondere, calculé en
# cellule 4). Les carreaux vides (ni habitants ni équipement) n'apportent rien
# aux cartes/calculs suivants.

population_grid_agglo=filtre_BPE_actifs (population_grid_agglo,land_use_data)


# Exemple d'usage : une carte pour un domaine
carte_ponderation_domaine(DOMAINES_BPE, population_grid_agglo, BPE_agglo, land_use_data, "D")

#pour exporter toutes les cartes en HTML :
for d in DOMAINES_BPE:
    carte = carte_ponderation_domaine(DOMAINES_BPE, population_grid_agglo, BPE_agglo, land_use_data, d)

    # Titre au-dessus de la carte : .explore() n'a pas de paramètre title, on
    # l'ajoute donc en HTML directement dans le document folium.
    titre_html = f'<h3 align="center" style="font-size:16px"><b>accessibilite_spatiale_{d}_{nom_reseau_str}</b></h3>'
    carte.get_root().html.add_child(folium.Element(titre_html))

    carte.save(
        os.path.join(output_path, f"ponderation_{d}_{nom_reseau_str}.html")
    )
  
# Affichage de la carte du domaine C (Enseignement) directement dans le notebook
from IPython.display import IFrame
IFrame(os.path.join(output_path, f"ponderation_C_{nom_reseau_str}.html"), width=900, height=600)

# Somme totale de la pondération (poids_gamme cumulé) pour chaque domaine BPE
ponderation_totale_par_domaine = [
    {"domaine": d, "nom": nom, "ponderation_totale": land_use_data_domaine(BPE_agglo, land_use_data, d)[d].sum()}
    for d, nom in DOMAINES_BPE.items()
]
ponderation_totale_par_domaine

In [ ]:
# Construction du réseau de transport multimodal, équivalent de setup_r5(data_path).
# En r5py, il n'y a pas de connexion séparée type r5r_core : l'objet TransportNetwork
# joue à la fois le rôle du réseau construit et du point d'entrée pour les calculs
# (ex: TravelTimeMatrixComputer, utilisé ensuite pour la matrice de temps de trajet).


print(data_path)
dir_tree(data_path)

In [ ]:

# Points d'origine/destination = centroïdes de la grille de population,
# équivalent de points <- fread(file.path(data_path, "poa_hexgrid.csv")).
# r5py exige une géométrie de type Point (pas les carreaux polygones bruts)
# et au moins une colonne "id", déjà présente dans population_grid_agglo.
points = population_grid_agglo[["id", "geometry"]].copy()
points["geometry"] = points.geometry.centroid

# On utilise date_JOB (calculé en cellule 2 à partir des dates de service
# réellement présentes dans le GTFS) plutôt que la date fixe de l'exemple R
# (13-05-2019) : cette dernière n'a aucune raison de tomber dans la période
# de validité de notre GTFS La Rochelle, ce qui ferait échouer le calcul
# transit (aucun service actif ce jour-là).
departure_datetime = datetime.datetime.strptime(date_JOB, "%Y%m%d").replace(
    hour=14, minute=0, second=0
)

ttm = r5py.TravelTimeMatrix(
    transport_network,
    origins=points,
    destinations=points,
    transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    departure=departure_datetime,
    max_time_walking=datetime.timedelta(minutes=30),
    max_time=datetime.timedelta(minutes=120),
)

# Sauvegarde de ttm (calcul long) pour pouvoir le recharger sans tout relancer
TTM_PATH = os.path.join(data_path, f"ttm_{nom_reseau_str}.parquet")
ttm.to_parquet(TTM_PATH, index=False)

ttm.head()

In [ ]:
# 3.2.1 Mesure des opportunités cumulées (cumulative_cutoff)
#
# Pas de package {accessibility} en Python : cumulative_cutoff() est un calcul
# simple (somme des opportunités atteignables sous un seuil de temps de trajet),
# reproduit ici directement en pandas plutôt que d'ajouter une dépendance.
#
# Contrairement à l'exemple R, pas besoin de renommer "travel_time_p50" :
# r5py.TravelTimeMatrix nomme déjà la colonne "travel_time" par défaut
# (percentiles=[50] étant la valeur par défaut).
#
# opportunity = "population" (pas de colonne "schools" dans nos données) :
# mesure le nombre d'habitants accessibles en <= cutoff minutes depuis chaque carreau.

# Pour éviter de refaire tourner la cellule ci-dessus (calcul ttm chronophage) :
# recharge ttm depuis le parquet sauvegardé.
TTM_PATH = os.path.join(data_path, f"ttm_{nom_reseau_str}.parquet")
ttm = pd.read_parquet(TTM_PATH)

cum_opportunities = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data,
    opportunity="population",
    travel_cost="travel_time",
    cutoff=30,
)

cum_opportunities.head()

In [ ]:

# 3.2.2 Coût de trajet minimum (cost_to_closest)

# Alias vers le land_use_data global (population), pris AVANT l'appel à
# cost_to_closest : le paramètre local du même nom de la fonction masquerait
# sinon la globale, rendant impossible la construction automatique via
# land_use_data_domaine() quand land_use_data n'est pas fourni.
# NB : cette ligne suppose que la cellule "Carroyage 200x200 INSEE" (qui définit
# land_use_data) a déjà été exécutée dans ce kernel.
_land_use_data_global = land_use_data

DOMAINE_CIBLE = "D"  # "D" = Santé et action sociale

min_time = cost_to_closest(
    land_use_data_domaine,
    BPE_agglo,
    _land_use_data_global,
    DOMAINES_BPE,
    ttm,
    opportunity=DOMAINE_CIBLE,
    travel_cost="travel_time",
)

min_time.head()


In [ ]:

# 3.2.3 Mesures de gravité (gravity)
#
# Gravité calculée par domaine BPE (DOMAINES_BPE, cellule 3.2.2), pondérée par
# gamme via land_use_data_domaine : la colonne "opportunity" est la
# pondération cumulée par gamme (gammes_poids) des équipements du domaine.
# Change DOMAINE_CIBLE_GRAVITY pour cibler un autre domaine
# (ex: "D" = santé, "C" = enseignement, "O" = tous les équipements).


DOMAINE_CIBLE_GRAVITY = "D"

negative_exp_grav = gravity(
    ttm,
    land_use_data=land_use_data_domaine(BPE_agglo, land_use_data, DOMAINE_CIBLE_GRAVITY),
    opportunity=DOMAINE_CIBLE_GRAVITY,
    travel_cost="travel_time",
    decay_function=decay_exponential(0.2),
)

negative_exp_grav.head()

In [ ]:
# 3.2.4 Mesures de compétition (floating_catchment_area)
#
# Pas d'implémentation fidèle du BFCA (Paez, Higgins & Vivona 2019) : c'est un
# algorithme d'équilibrage itératif offre/demande, trop spécifique pour être
# reconstitué de mémoire avec certitude. À la place : Enhanced 2SFCA
# (Luo & Qi, 2009), une méthode de compétition bien établie et proche en
# esprit (ratio offre/demande pondéré par la décroissance, en deux étapes,
# sans boucle d'équilibrage). Les valeurs ne correspondront pas exactement
# à celles de {accessibility}::floating_catchment_area(method = "bfca").



# opportunity = domaine BPE (cohérent avec 3.2.2/3.2.3) : la colonne "supply"
# vient de land_use_data_domaine(DOMAINE_CIBLE_COMPETITION) (pondération
# cumulée par gamme), fusionnée avec "population" (demande) de land_use_data.
# enhanced_2sfca() elle-même ne change pas : générique sur land_use_data/opportunity.
DOMAINE_CIBLE_COMPETITION = "D"  # "D" = Santé et action sociale, ou "O" = tous

land_use_data_competition = land_use_data_domaine(BPE_agglo, land_use_data, DOMAINE_CIBLE_COMPETITION).merge(
    land_use_data[["id", "population"]], on="id", how="left"
)

e2sfca_competition = enhanced_2sfca(
    ttm,
    land_use_data=land_use_data_competition,
    opportunity=DOMAINE_CIBLE_COMPETITION,
    travel_cost="travel_time",
    demand="population",
    decay_function=decay_exponential(0.05),
)

e2sfca_competition.head()

In [ ]:
#3.2.5 Comparaison des seuils <= vs < (cutoff)
#
# r5py 1.1.7 n'a pas d'équivalent à r5r::accessibility() : seuls TransportNetwork,
# TravelTimeMatrix, Isochrones et DetailedItineraries sont exportés (r5py/__init__.py).
# Il n'y a donc qu'un seul chemin de calcul disponible ici (celui de 3.2.1), pas deux
# méthodes à comparer entre elles. On illustre malgré tout le point pédagogique du texte :
# r5r::accessibility() ne compte que les trajets strictement sous le seuil (< cutoff),
# alors que notre cumulative_cutoff() (comme accessibility::cumulative_cutoff() en R)
# compte les trajets <= cutoff. cumulative_cutoff(cutoff=29) équivaut donc à un seuil
# "moins de 30 minutes".
DOMAINE_CIBLE_CUTOFF = "A"  # "D" = Santé et action sociale, ou "O" = tous

land_use_data_cutoff = land_use_data_domaine(BPE_agglo, land_use_data, DOMAINE_CIBLE_CUTOFF)

cum_cutoff_30 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data_cutoff,
    opportunity=DOMAINE_CIBLE_CUTOFF,
    travel_cost="travel_time",
    cutoff=30,
)

cum_cutoff_29 = cumulative_cutoff(
    ttm,
    land_use_data=land_use_data_cutoff,
    opportunity=DOMAINE_CIBLE_CUTOFF,
    travel_cost="travel_time",
    cutoff=29,
)

cutoff_comparison = cum_cutoff_30.merge(
    cum_cutoff_29, on="id", suffixes=("_cutoff_30_inclus", "_cutoff_29_soit_moins_de_30min")
)

cutoff_comparison.head()

In [ ]:

# 3.3.1 Distribution spatiale de l'accessibilité urbaine
#
# Pas d'équivalent {aopdata}::read_grid("Porto Alegre") : on réutilise directement
# le carroyage population_grid_cda (chargé en cellule 2), qui est déjà la grille
# utilisée comme origines/destinations du calcul (cellule 5) — pas besoin de
# filtrer sur les id de "points", ils correspondent déjà.
#
# Accessibilité affichée : cum_cutoff_30 (cellule 3.2.5), sur le domaine BPE
# DOMAINE_CIBLE_CUTOFF (cohérent avec 3.2.2-3.2.5), pas figé sur "schools".

# Fond de carte pour l'export HTML interactif ci-dessous : change FOND_CARTE
# pour choisir parmi les trois options.
FONDS_CARTE = {
    "OpenStreetMap": "OpenStreetMap",
    "CartoDB Positron": "CartoDB positron",
    "CartoDB Dark Matter": "CartoDB dark_matter",
}
FOND_CARTE = "OpenStreetMap"  # "OpenStreetMap" / "CartoDB Positron" / "CartoDB Dark Matter"

for d in {"O", "A", "B", "C", "D", "E", "F","G"}:
    DOMAINE_CIBLE_CUTOFF=d 
    nom_domaine_cutoff = DOMAINES_BPE.get(DOMAINE_CIBLE_CUTOFF, DOMAINE_CIBLE_CUTOFF)

    # cum_cutoff_30 doit être recalculé pour chaque domaine d : sinon .plot(column=d)
    # échoue dès que d diffère du domaine utilisé lors du dernier calcul (cellule 3.2.5).
    cum_cutoff_30 = cumulative_cutoff(
        ttm,
        land_use_data=land_use_data_domaine(BPE_agglo, land_use_data, DOMAINE_CIBLE_CUTOFF),
        opportunity=DOMAINE_CIBLE_CUTOFF,
        travel_cost="travel_time",
        cutoff=30,
    )

    spatial_access = population_grid_agglo[["id", "geometry"]].merge(cum_cutoff_30, on="id")

    # contextily fournit les tuiles en Web Mercator (EPSG:3857) : on reprojette
    # juste pour l'affichage, spatial_access (en EPSG:2154) n'est pas modifié.
    spatial_access_3857 = spatial_access.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(8, 8))
    spatial_access_3857.plot(
        column=DOMAINE_CIBLE_CUTOFF,
        cmap="inferno",
        alpha=0.7,
        legend=True,
        edgecolor="none",
        ax=ax,
        legend_kwds={"label": f"{nom_domaine_cutoff}\n(pondéré)"},
    )
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=spatial_access_3857.crs)
    ax.set_axis_off()

    # fig.suptitle() plutôt que ax.set_title() : ax.set_title() centre le texte
    # sur la seule largeur de l'axe carte, qui est plus étroit que la figure une
    # fois la colorbar de legend=True ajoutée à droite — un titre un peu long
    # débordait donc sur l'échelle. suptitle() centre sur toute la figure.
    fig.suptitle(
        f"Distribution spatiale de l'accessibilité – {nom_domaine_cutoff} (30 min)",
        fontsize=12,
    )

    plt.savefig(
        os.path.join(output_path, f"accessibilite_spatiale_{d}_{nom_reseau_str}.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    # Export HTML interactif équivalent, avec le fond de carte choisi ci-dessus
    # (contours des carreaux transparents, cf. carte_ponderation_domaine).
    carte_interactive = spatial_access.explore(
        column=DOMAINE_CIBLE_CUTOFF,
        cmap="inferno",
        tiles=FONDS_CARTE[FOND_CARTE],
        legend=True,
        legend_kwds={"caption": f"{nom_domaine_cutoff} (pondéré)"},
        style_kwds={"weight": 0, "opacity": 0},
    )

    # Titre au-dessus de la carte interactive : .explore() n'a pas de paramètre
    # title, on l'ajoute donc en HTML directement dans le document folium.
    titre_html = f'<h3 align="center" style="font-size:16px"><b>Accessibilité {nom_domaine_cutoff} 30 min</b></h3>'
    carte_interactive.get_root().html.add_child(folium.Element(titre_html))

    carte_interactive.save(
        os.path.join(output_path, f"accessibilite_spatiale_{d}_{nom_reseau_str}.html")
    )